In [7]:
import numpy as np
from fractions import Fraction
from helpers import gcd_decimal, clear_denominators
import itertools

In [5]:
def rescale_coefficients(a, u):
    a = np.array(a)
    a = a.astype(float)
    indices = np.argsort(a)
    a_sorted = a[indices]
    for i, a_i in enumerate(a_sorted):
        if a_i + a_sorted[0] > u:
            a_sorted[i:] = u
            a[indices[i:]] = u
    # restrict to non-trivial elements
    cutoff = next((i for i in range(len(a)) if a[i] >= u), 0)
    b = a[:cutoff] # only keeping elements which are smaller than u
    if len(b) == 0:
        return a[indices]
    # compute all feasible sums to compare:
    all_feasible_tuples = compute_feasible_tuples(b, u) # [{n-tuples of indices of numbers that form feasible sums} for order n]
    def choose_new_values(a_i, u):
        ## this could be extended to a much larger list with high values first
        quotient = u / a_i
        if int(quotient) == quotient:
            quotients = [int(quotient) - 1]
        else:
            quotients = [int(quotient), int(quotient) + 1]
        return [Fraction(u, quotient) for quotient in quotients]
    keep_trying = True
    while keep_trying:
        keep_trying = False
        i, gcd_i = min(list(enumerate(gcd_decimal(a_i, u) for a_i in b)), key=lambda x: x[1])
        if gcd_i == u:
            break
        print(i, b[i])
        feasible_i_tuples = [
            {
                ntuple for ntuple in tuple_set if i in ntuple
                }
            for tuple_set in all_feasible_tuples
        ]
        new_values = choose_new_values(b[i], u)
        for value in new_values:
            feasible_i_tuples_for_value = compute_feasible_i_tuples(i, value, b, u)
            if feasible_i_tuples_for_value == feasible_i_tuples:
                print('accepting value ', float(value))
                a[indices[i]] = value
                indices = np.argsort(a)
                a_sorted = a[indices]
                # cutoff = next((i for i in range(len(a)) if a[i] >= u), 0) I think this is not needed, as the number of smaller than u elements should be the same
                b = a[:cutoff] # only keeping elements which are smaller than u
                keep_trying = True
                break
    return a

def k_sum_less_than(arr, k, T):
    """
    Return all k-tuples of indices (i1, ..., ik) such that
    arr[i1] + ... + arr[ik] < T.
    `arr` must be sorted.
    Supports k >= 1.
    """
    n = len(arr)
    if k < 1:
        raise ValueError("k must be >= 1")
    if k > n:
        return []

    def helper(start, k, T):
        # Base case: k = 1 → all indices i where arr[i] < T
        if k == 1:
            too_high_index = next((i for i in range(start, n) if arr[i] > T), start)
            return [(i,) for i in range(start, too_high_index)]

        # Base case: k = 2 → classical two-pointer
        if k == 2:
            left, right = start, n - 1
            out = []

            while left < right:
                s = arr[left] + arr[right]
                if s <= T:
                    # all (left, i) for i in [left+1..right]
                    out.extend([(left, i) for i in range(left+1, right+1)])
                    left += 1
                else:
                    right -= 1
            return out

        # Recursive case: k > 2
        out = []
        for i in range(start, n - k + 1):

            # Prune: smallest possible sum too large
            min_possible = arr[i] + sum(arr[i+1 : i+k])
            if min_possible > T:
                break

            # Prune: even the largest possible sum is small enough
            max_possible = arr[i] + sum(arr[-(k-1):])
            if max_possible <= T:
                from itertools import combinations
                for combo in combinations(range(i+1, n), k-1):
                    out.append((i,) + combo)
                continue

            # Recursive descent
            sub = helper(i+1, k-1, T - arr[i])
            for tup in sub:
                out.append((i,) + tup)

        return out

    return helper(0, k, T)


def compute_sums_feasible(i, a_i, b, u):
    csum = np.cumsum(b)
    max_order = np.searchsorted(csum, u, side='right')
    largest_index_per_order = {2: len(b) - 1, **{order: next((j - 1 for j, a_i in enumerate(b[order - 1:]) if a_i > u - csum[order - 2]), len(b) - order) + order - 1 for order in range(3, max_order + 1)}}
    sums_feasible = [
        {
            tuple(sorted((i,) + ntuple)) for ntuple in itertools.combinations(
                (j for j in range(largest_index_per_order[order] + 1) if j != i),
                order - 1) if sum(b[j] for j in ntuple) <= u - a_i
            } for order in range(2, max_order + 1)
    ] 
    return sums_feasible

def compute_feasible_i_tuples(i, value, b, u):
    csum = np.cumsum(b)
    max_order = np.searchsorted(csum, u, side='right')
    feasible_sums = [
        {
            tuple(sorted((i,) + tuple(index if index < i else index + 1 for index in ntuple))) for ntuple in k_sum_less_than([b_j for j, b_j in enumerate(b) if j != i], order - 1, u - value) 
            } for order in range(2, max_order + 1)
    ]
    return feasible_sums

def compute_feasible_tuples(b, u):
    csum = np.cumsum(b)
    max_order = np.searchsorted(csum, u, side='right')
    largest_index_per_order = {**{order: next((j - 1 for j, a_i in enumerate(b[order - 1:]) if a_i > u - csum[order - 2]), len(b) - order) + order - 1 for order in range(2, max_order + 1)}}
    feasible_sums = [
        {
            ntuple for ntuple in k_sum_less_than(b[:largest_index_per_order[order] + 1], order, u) 
            } for order in range(3, max_order + 1)
    ]
    return feasible_sums

In [13]:
a = [545.6182548145297, 353.6978281098691, 576.328346014674, 554.074624676238]
u = 1809.919053615311
a_new = rescale_coefficients(a, u)
print(a_new)
clear_denominators(a_new, u)

[353.69782811 545.61825481 554.07462468 576.32834601]


([118765887591, 183209596353, 186049105653, 193521537647], 607768098800)

In [1]:
if not {}:
    print(1)

1
